# Load Serverless Product Rates

This notebook creates the `serverless_product_rates` table for simple serverless products:
- Vector Search
- Model Serving (CPU/GPU)
- AI Gateway
- Agent Evaluation
- Shutterstock ImageAI

**Source:** https://docs.databricks.com/aws/en/resources/pricing


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_SERVERLESS_RATES = f"{CATALOG}.{SCHEMA}.serverless_product_rates"

print(f"✅ Target table: {TABLE_SERVERLESS_RATES}")


In [ ]:
import pandas as pd
from datetime import datetime

# =============================================================================
# SERVERLESS PRODUCT RATES
# Formula: 
#   if is_hourly: DBU = CEIL(input / input_divisor) * dbu_rate * hours
#   else: DBU = (input / input_divisor) * dbu_rate
#
# NOTE: Creating one row per cloud (AWS, AZURE, GCP) for future flexibility
# =============================================================================

CLOUDS = ["AWS", "AZURE", "GCP"]

# Base rates (same across clouds for now, but structure allows future changes)
BASE_RATES = [
    # =========================================================================
    # VECTOR SEARCH
    # =========================================================================
    {
        "product": "vector_search",
        "size_or_model": "standard",
        "rate_type": "hourly",
        "dbu_rate": 4.0,
        "input_divisor": 2000000,  # 2M vectors per unit
        "is_hourly": True,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "Standard endpoint - 2M vectors per unit",
    },
    {
        "product": "vector_search",
        "size_or_model": "storage_optimized",
        "rate_type": "hourly",
        "dbu_rate": 18.29,
        "input_divisor": 64000000,  # 64M vectors per unit
        "is_hourly": True,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "Storage optimized - 64M vectors per unit",
    },
    
    # =========================================================================
    # MODEL SERVING - CPU
    # =========================================================================
    {
        "product": "model_serving",
        "size_or_model": "cpu",
        "rate_type": "hourly",
        "dbu_rate": 1.0,
        "input_divisor": 1,  # per concurrent request
        "is_hourly": True,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "1 concurrent request/hr = 1 DBU/hr",
    },
    
    # =========================================================================
    # MODEL SERVING - GPU (NOTE: GPU rates differ by cloud, handled separately below)
    # =========================================================================
    
    # =========================================================================
    # AI GATEWAY
    # =========================================================================
    {
        "product": "ai_gateway",
        "size_or_model": "guardrails",
        "rate_type": "per_token",
        "dbu_rate": 21.429,
        "input_divisor": 1000000,  # per 1M tokens
        "is_hourly": False,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "AI Guardrails - 21.429 DBU per 1M tokens",
    },
    {
        "product": "ai_gateway",
        "size_or_model": "inference_tables",
        "rate_type": "per_gb",
        "dbu_rate": 7.143,
        "input_divisor": 1,  # per GB
        "is_hourly": False,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "Inference Tables - 7.143 DBU per GB payload",
    },
    {
        "product": "ai_gateway",
        "size_or_model": "usage_tracking",
        "rate_type": "per_gb",
        "dbu_rate": 1.429,
        "input_divisor": 1,  # per GB
        "is_hourly": False,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "Usage Tracking - 1.429 DBU per GB payload",
    },
    
    # =========================================================================
    # OTHER
    # =========================================================================
    {
        "product": "image_ai",
        "size_or_model": "shutterstock",
        "rate_type": "per_image",
        "dbu_rate": 0.857,
        "input_divisor": 1,  # per image
        "is_hourly": False,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "Shutterstock ImageAI - 0.857 DBU per image",
    },
    {
        "product": "agent_evaluation",
        "size_or_model": "judge",
        "rate_type": "per_request",
        "dbu_rate": 1.0,
        "input_divisor": 1,  # per request
        "is_hourly": False,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": "Agent Evaluation - 1 DBU per judge request",
    },
]

# Expand base rates to all clouds
SERVERLESS_RATES = []
for cloud in CLOUDS:
    for rate in BASE_RATES:
        row = rate.copy()
        row["cloud"] = cloud
        SERVERLESS_RATES.append(row)

# =============================================================================
# CLOUD-SPECIFIC GPU MODEL SERVING RATES
# AWS, Azure, and GCP have different GPU configurations
# =============================================================================

# AWS GPU Model Serving
AWS_GPU_RATES = [
    {"size_or_model": "gpu_small_t4", "dbu_rate": 10.48, "description": "Small - T4 or equivalent"},
    {"size_or_model": "gpu_medium_a10g_1x", "dbu_rate": 20.0, "description": "Medium - A10G x 1GPU"},
    {"size_or_model": "gpu_medium_a10g_4x", "dbu_rate": 112.0, "description": "Medium 4X - A10G x 4GPU"},
    {"size_or_model": "gpu_medium_a10g_8x", "dbu_rate": 290.8, "description": "Medium 8X - A10G x 8GPU"},
    {"size_or_model": "gpu_xlarge_a100_40gb_8x", "dbu_rate": 538.4, "description": "XLarge - A100 40GB x 8GPU"},
    {"size_or_model": "gpu_xlarge_a100_80gb_8x", "dbu_rate": 628.0, "description": "XLarge - A100 80GB x 8GPU"},
]

# Azure GPU Model Serving
AZURE_GPU_RATES = [
    {"size_or_model": "gpu_small_t4", "dbu_rate": 10.48, "description": "Small - T4 or equivalent"},
    {"size_or_model": "gpu_xlarge_a100_80gb_1x", "dbu_rate": 78.6, "description": "XLarge - A100 80GB x 1GPU"},
    {"size_or_model": "gpu_2xlarge_a100_80gb_2x", "dbu_rate": 157.2, "description": "2XLarge - A100 80GB x 2GPU"},
    {"size_or_model": "gpu_4xlarge_a100_80gb_4x", "dbu_rate": 314.4, "description": "4XLarge - A100 80GB x 4GPU"},
]

# GCP GPU Model Serving
GCP_GPU_RATES = [
    {"size_or_model": "gpu_medium_g2_standard_8", "dbu_rate": 5.0, "description": "Medium - G2 Standard 8 x 1GPU"},
]

# Add cloud-specific GPU rates
for gpu_rate in AWS_GPU_RATES:
    SERVERLESS_RATES.append({
        "cloud": "AWS",
        "product": "model_serving",
        "size_or_model": gpu_rate["size_or_model"],
        "rate_type": "hourly",
        "dbu_rate": gpu_rate["dbu_rate"],
        "input_divisor": 1,
        "is_hourly": True,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": gpu_rate["description"],
    })

for gpu_rate in AZURE_GPU_RATES:
    SERVERLESS_RATES.append({
        "cloud": "AZURE",
        "product": "model_serving",
        "size_or_model": gpu_rate["size_or_model"],
        "rate_type": "hourly",
        "dbu_rate": gpu_rate["dbu_rate"],
        "input_divisor": 1,
        "is_hourly": True,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": gpu_rate["description"],
    })

for gpu_rate in GCP_GPU_RATES:
    SERVERLESS_RATES.append({
        "cloud": "GCP",
        "product": "model_serving",
        "size_or_model": gpu_rate["size_or_model"],
        "rate_type": "hourly",
        "dbu_rate": gpu_rate["dbu_rate"],
        "input_divisor": 1,
        "is_hourly": True,
        "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        "description": gpu_rate["description"],
    })

print(f"✅ Defined {len(SERVERLESS_RATES)} serverless product rates")
print(f"   - {len(BASE_RATES)} base rates × 3 clouds = {len(BASE_RATES) * 3}")
print(f"   - AWS GPU: {len(AWS_GPU_RATES)}, Azure GPU: {len(AZURE_GPU_RATES)}, GCP GPU: {len(GCP_GPU_RATES)}")


In [ ]:
# =============================================================================
# CREATE DATAFRAME AND SAVE
# =============================================================================

# Add timestamp
for row in SERVERLESS_RATES:
    row["updated_at"] = datetime.utcnow().isoformat()

df = pd.DataFrame(SERVERLESS_RATES)

print(f"📊 Products: {df['product'].unique().tolist()}")
print(f"📊 Total rates: {len(df)}")
display(df)

# Save to table
spark_df = spark.createDataFrame(df)
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_SERVERLESS_RATES)

print(f"\n✅ Saved to {TABLE_SERVERLESS_RATES}")
